In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Tuple
import math
import numpy as np

# -----------------------------
# Constants
# -----------------------------
SIGMA = 5.670374419e-8  # Stefan-Boltzmann (W m-2 K-4)
KAPPA = 0.4             # von Karman
G = 9.80665             # m s-2
R_AIR = 287.05          # J kg-1 K-1
CP_AIR = 1004.0         # J kg-1 K-1
EPS_SNOW = 0.99

L_SUB = 2.834e6         # J kg-1 (sublimation)
RHO_ICE = 917.0         # kg m-3

# -----------------------------
# Helper: saturation vapor pressure over ice (Pa)
# (Murphy & Koop 2005-ish simple approx)
# -----------------------------
def esat_ice_pa(Tk: float) -> float:
    # Valid for typical cold conditions; good enough for toy model.
    # Tk in Kelvin.
    Tc = Tk - 273.15
    return 611.15 * math.exp(22.452 * Tc / (Tc + 272.55))

def qsat_ice(Tk: float, p_pa: float) -> float:
    e = esat_ice_pa(Tk)
    return 0.622 * e / max(1e-6, (p_pa - e))

def air_density(p_pa: float, Tk: float) -> float:
    return p_pa / (R_AIR * Tk)

# -----------------------------
# Simple snow thermal conductivity k(rho)
# Many formulas exist; use a smooth monotonic toy law.
# -----------------------------
def k_snow(rho: float) -> float:
    # W m-1 K-1
    # Rough range: ~0.05 (very light) up to ~0.5 (dense)
    x = min(max(rho / 500.0, 0.0), 2.0)
    return 0.05 + 0.25 * x**2

def heat_capacity_snow(rho: float) -> float:
    # Volumetric heat capacity (rho * c), J m-3 K-1
    c = 2100.0  # J kg-1 K-1 (ice approx; snow similar order)
    return rho * c

# -----------------------------
# Tridiagonal solver (Thomas algorithm)
# Solves Ax = d, where A has lower a, diag b, upper c.
# a,b,c,d are 1D arrays length n; a[0]=0, c[-1]=0.
# -----------------------------
def solve_tridiagonal(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:
    n = len(d)
    cp = np.zeros(n)
    dp = np.zeros(n)
    x  = np.zeros(n)

    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    for i in range(1, n):
        denom = b[i] - a[i] * cp[i-1]
        cp[i] = c[i] / denom if i < n-1 else 0.0
        dp[i] = (d[i] - a[i]*dp[i-1]) / denom

    x[-1] = dp[-1]
    for i in range(n-2, -1, -1):
        x[i] = dp[i] - cp[i]*x[i+1]
    return x

# -----------------------------
# Data structures
# -----------------------------
@dataclass
class Forcing1H:
    # All in SI
    Ta: float        # air temperature (K)
    RH: float        # relative humidity (0-1)
    p: float         # pressure (Pa)
    U10: float       # wind speed at 10m (m s-1)
    SWd: float       # downwelling shortwave (W m-2)
    LWd: float       # downwelling longwave (W m-2)
    Psnow: float     # snowfall rate (kg m-2 s-1) (water-equivalent mass flux)
    Tsoil: float     # soil temp at snow/soil interface (K)

@dataclass
class Layer:
    # Stratigraphic layer (mass/structure). Thermal profile handled separately by grid.
    swe: float        # kg m-2  (water equivalent mass per area)
    rho: float        # kg m-3  (layer bulk density)
    x_ws: float = 0.0 # [0,1] wind slabness
    x_dh: float = 0.0 # [0,1] depth hoarness
    age_s: float = 0.0# seconds since creation (for erodability etc.)

    @property
    def thickness(self) -> float:
        # thickness = SWE / rho (since SWE [kg/m2] corresponds to water mass; snow mass is SWE (same mass), volume=mass/rho
        return self.swe / max(1e-6, self.rho)

# -----------------------------
# Snowpack model
# -----------------------------
class ArcticSnowToy:
    def __init__(
        self,
        # thermal grid config
        dz_template: List[float],
        # compaction/target density params (6-8)
        rho_min: float = 100.0,
        rho_ref: float = 280.0,
        rho_max: float = 520.0,
        sigma0: float = 400.0,          # Pa
        dR_ws: float = 160.0,           # kg m-3
        dR_dh: float = 100.0,           # kg m-3
        tau0_days: float = 6.0,         # days
        a_sigma: float = 1.0,
        a_ws: float = 2.0,
        a_dh: float = 2.0,
        # DH / WS diagnostics
        G0: float = 10.0,               # K m-1
        tau_dh_days: float = 6.0,
        tau_ws_days: float = 1.0,
        # transport threshold
        Ut0: float = 6.0,               # m/s at 10m baseline
        Ut_slope: float = -0.05,        # m/s per K (toy)
        T0: float = 263.15,             # K ref
        p_ws: float = 3.0,
        # surface parameters
        albedo_fresh: float = 0.85,
        albedo_old: float = 0.75,
        albedo_tau_days: float = 10.0,
        z0: float = 1e-3,
        z_ref: float = 10.0,
        # layer management
        Nmax_layers: int = 8,
        min_new_layer_swe: float = 2.0, # kg m-2 (≈2 mm SWE)
    ):
        self.dz_template = np.array(dz_template, dtype=float)

        # compaction params
        self.rho_min = rho_min
        self.rho_ref = rho_ref
        self.rho_max = rho_max
        self.sigma0  = sigma0
        self.dR_ws   = dR_ws
        self.dR_dh   = dR_dh
        self.tau0    = tau0_days * 86400.0
        self.a_sigma = a_sigma
        self.a_ws    = a_ws
        self.a_dh    = a_dh

        # regime params
        self.G0 = G0
        self.tau_dh = tau_dh_days * 86400.0
        self.tau_ws = tau_ws_days * 86400.0
        self.Ut0 = Ut0
        self.Ut_slope = Ut_slope
        self.T0 = T0
        self.p_ws = p_ws

        # surface
        self.albedo_fresh = albedo_fresh
        self.albedo_old   = albedo_old
        self.albedo_tau   = albedo_tau_days * 86400.0
        self.z0 = z0
        self.z_ref = z_ref

        # layers
        self.layers: List[Layer] = []
        self.Nmax_layers = Nmax_layers
        self.min_new_layer_swe = min_new_layer_swe

        # thermal state
        self.Tgrid: Optional[np.ndarray] = None
        self.zgrid: Optional[np.ndarray] = None
        self.dzgrid: Optional[np.ndarray] = None
        self.Ts: float = 258.15  # surface temperature (K), initialized
        self.albedo: float = albedo_fresh
        self.time_s: float = 0.0

    # ---------
    # Geometry helpers
    # ---------
    def total_swe(self) -> float:
        return sum(L.swe for L in self.layers)

    def total_height(self) -> float:
        return sum(L.thickness for L in self.layers)

    def ensure_thermal_grid(self, H: float, Tsoil: float):
        """
        Build/adjust thermal grid to match current snow height H.
        Uses dz_template from surface downward, truncating to H.
        Initializes Tgrid if needed by linear profile between Ts and Tsoil.
        """
        if H <= 1e-6:
            self.Tgrid = None
            self.zgrid = None
            self.dzgrid = None
            return

        dzs = []
        remaining = H
        for dz in self.dz_template:
            if remaining <= 0:
                break
            dzs.append(min(dz, remaining))
            remaining -= dz
        dzs = np.array(dzs, dtype=float)

        zc = np.cumsum(dzs) - 0.5 * dzs  # cell centers from surface downward
        # store as depth from surface; basal is at depth H.

        if (self.Tgrid is None) or (len(self.Tgrid) != len(dzs)):
            # initialize temperature profile: linear in depth between Ts and Tsoil
            Ttop = self.Ts
            Tbot = Tsoil
            self.Tgrid = Ttop + (Tbot - Ttop) * (zc / max(1e-6, H))
        else:
            # simple remap if dz count changed (toy): interpolate old profile onto new centers
            old_zc = self.zgrid
            old_T  = self.Tgrid
            self.Tgrid = np.interp(zc, old_zc, old_T, left=old_T[0], right=old_T[-1])

        self.zgrid = zc
        self.dzgrid = dzs

    # ---------
    # Surface exchange coefficients (neutral; add stability later if desired)
    # ---------
    def bulk_coeff(self) -> float:
        # neutral transfer coefficient ~ (kappa / ln(z/z0))^2
        L = math.log(self.z_ref / self.z0)
        return (KAPPA / L) ** 2

    # ---------
    # Wind transport threshold
    # ---------
    def Ut(self, Tair: float) -> float:
        return max(1.0, self.Ut0 + self.Ut_slope * (Tair - self.T0))

    # ---------
    # SEB residual F(Ts)=0
    # ---------
    def seb_residual(self, Ts: float, forc: Forcing1H, k_half: float, T1: float, dz_half: float) -> Tuple[float, float, float, float, float]:
        """
        Returns residual and fluxes: (F, H, LE, G, Rn)
        Positive fluxes warm the surface.
        """
        rho_a = air_density(forc.p, forc.Ta)
        C = self.bulk_coeff()
        U = max(0.1, forc.U10)

        # humidity
        qa = forc.RH * qsat_ice(forc.Ta, forc.p)
        qs = qsat_ice(Ts, forc.p)

        # turbulent
        H = rho_a * CP_AIR * C * U * (forc.Ta - Ts)
        LE = rho_a * L_SUB * C * U * (qa - qs)

        # radiation
        Rn = forc.SWd * (1.0 - self.albedo) + forc.LWd - EPS_SNOW * SIGMA * Ts**4

        # conductive into snow (positive warms surface if snow warmer than surface)
        Gflux = -k_half * (T1 - Ts) / max(1e-6, dz_half)

        F = Rn + H + LE + Gflux
        return F, H, LE, Gflux, Rn

    def solve_Ts(self, forc: Forcing1H) -> float:
        """
        Solve for surface temperature Ts from SEB with Newton + bisection fallback.
        Constrains Ts <= 273.15 K (melt condition handled outside).
        """
        H = self.total_height()
        if H <= 1e-6:
            # no snow: just set Ts to soil or air (toy)
            return min(forc.Ta, forc.Tsoil)

        # thermal coupling to first cell
        T1 = float(self.Tgrid[0])
        rho1 = self.estimate_surface_density()
        k1 = k_snow(rho1)
        k_half = k1
        dz_half = 0.5 * float(self.dzgrid[0])

        # bracket
        Tmin = max(180.0, forc.Ta - 50.0)
        Tmax = 273.15  # constrain to melting point

        # initial guess
        Ts = float(np.clip(self.Ts, Tmin, Tmax))

        def Fval(x):
            F, *_ = self.seb_residual(x, forc, k_half, T1, dz_half)
            return F

        # Bisection bracket search
        a, b = Tmin, Tmax
        Fa, Fb = Fval(a), Fval(b)

        # Ensure sign change; if not, expand Tmin a bit or accept closest
        if Fa * Fb > 0:
            # choose Ts that minimizes |F|
            candidates = [a, b, Ts]
            best = min(candidates, key=lambda x: abs(Fval(x)))
            return best

        # Newton iterations with bisection safeguard
        for _ in range(20):
            F, Hflx, LEflx, Gflx, Rn = self.seb_residual(Ts, forc, k_half, T1, dz_half)

            if abs(F) < 0.5:  # W m-2 tolerance
                return float(np.clip(Ts, Tmin, Tmax))

            # numerical derivative dF/dTs
            eps = 0.1
            Fp = Fval(min(Tmax, Ts + eps))
            Fm = Fval(max(Tmin, Ts - eps))
            dF = (Fp - Fm) / (2 * eps)
            if abs(dF) < 1e-6:
                break

            Ts_new = Ts - F / dF

            # keep in bracket by bisection if needed
            if Ts_new <= a or Ts_new >= b or not np.isfinite(Ts_new):
                Ts_new = 0.5 * (a + b)

            # update bracket
            Fnew = Fval(Ts_new)
            if Fa * Fnew <= 0:
                b, Fb = Ts_new, Fnew
            else:
                a, Fa = Ts_new, Fnew

            Ts = Ts_new

        # fallback pure bisection
        for _ in range(50):
            m = 0.5 * (a + b)
            Fm = Fval(m)
            if abs(Fm) < 0.5:
                return float(m)
            if Fa * Fm <= 0:
                b, Fb = m, Fm
            else:
                a, Fa = m, Fm
        return float(0.5 * (a + b))

    def estimate_surface_density(self) -> float:
        # Use topmost stratigraphic layer density if exists
        if not self.layers:
            return self.rho_min
        return float(self.layers[-1].rho)

    # ---------
    # Implicit conduction step (Dirichlet Ts at top, Tsoil at bottom)
    # ---------
    def conduction_step(self, Ts: float, Tsoil: float, dt: float):
        if self.Tgrid is None:
            return
        n = len(self.Tgrid)
        dz = self.dzgrid

        # build k and cvol arrays per cell
        # Map stratigraphic densities onto thermal grid crudely: use surface density everywhere for toy,
        # or compute depth-dependent rho by distributing layers. We'll do a simple depth mapping.
        rho_z = self.rho_profile_on_thermal_grid()
        k = np.array([k_snow(r) for r in rho_z], dtype=float)
        cvol = np.array([heat_capacity_snow(r) for r in rho_z], dtype=float)

        # harmonic mean at interfaces
        def kh(i, j):
            return 2 * k[i] * k[j] / max(1e-12, (k[i] + k[j]))

        # assemble tridiagonal system for unknown T^{n+1}
        a = np.zeros(n)
        b = np.zeros(n)
        c = np.zeros(n)
        d = np.zeros(n)

        # Top Dirichlet (surface): impose T0 = Ts by setting first equation strongly
        # but since centers are not at the boundary, we enforce via ghost approach:
        # Simplest: set first cell with boundary flux using Ts. We'll include Ts in RHS.
        # Use finite-volume: interface at top between Ts boundary and cell 0.
        for i in range(n):
            dz_i = dz[i]
            vol = dz_i  # per unit area
            b[i] = cvol[i] * vol / dt
            d[i] = b[i] * self.Tgrid[i]

        # Add diffusion terms
        for i in range(n):
            dz_i = dz[i]

            # upper interface
            if i == 0:
                # boundary at surface: between Ts and cell 0 center
                dz_up = 0.5 * dz[i]
                k_up = k[i]  # boundary uses cell k
                coeff = k_up / dz_up
                b[i] += coeff
                d[i] += coeff * Ts
            else:
                dz_up = 0.5 * (dz[i-1] + dz[i])
                k_up = kh(i-1, i)
                coeff = k_up / dz_up
                b[i] += coeff
                a[i] -= coeff

            # lower interface
            if i == n - 1:
                dz_dn = 0.5 * dz[i]
                k_dn = k[i]
                coeff = k_dn / dz_dn
                b[i] += coeff
                d[i] += coeff * Tsoil
            else:
                dz_dn = 0.5 * (dz[i] + dz[i+1])
                k_dn = kh(i, i+1)
                coeff = k_dn / dz_dn
                b[i] += coeff
                c[i] -= coeff

        # Convert a,c to lower/upper with signs
        # We built a[i] negative, c[i] negative; turn them into positive lower/upper magnitudes:
        lower = -a
        upper = -c
        diag = b
        rhs = d

        Tnew = solve_tridiagonal(lower, diag, upper, rhs)
        self.Tgrid = Tnew

    def rho_profile_on_thermal_grid(self) -> np.ndarray:
        """
        Distribute stratigraphic layer densities onto thermal grid by depth from surface.
        Simple piecewise constant mapping.
        """
        n = len(self.Tgrid)
        rho_z = np.full(n, self.rho_min, dtype=float)
        if not self.layers:
            return rho_z

        # build depth intervals for layers from surface downward
        # layers list is bottom->top? We used layers append at top; so layers[0] bottom, layers[-1] top.
        # We'll create from surface downward => traverse reversed.
        layer_th = [L.thickness for L in self.layers]
        layer_rho = [L.rho for L in self.layers]
        # reverse to surface-down
        th_surf = list(reversed(layer_th))
        rho_surf = list(reversed(layer_rho))

        # cell centers depth from surface:
        zc = self.zgrid
        # assign
        cum = 0.0
        idx_layer = 0
        for i in range(n):
            z = zc[i]
            while idx_layer < len(th_surf) and z > cum + th_surf[idx_layer]:
                cum += th_surf[idx_layer]
                idx_layer += 1
            if idx_layer >= len(th_surf):
                rho_z[i] = rho_surf[-1]
            else:
                rho_z[i] = rho_surf[idx_layer]
        return rho_z

    # ---------
    # DH / WS update
    # ---------
    def gradients_local(self) -> Tuple[float, float]:
        """
        Return (G_surf, G_basal) in K/m based on thermal grid.
        surface: between Ts boundary and first cell center
        basal: between soil boundary and ~10 cm above ground (depth from surface H-0.10)
        """
        H = self.total_height()
        if self.Tgrid is None or H <= 1e-6:
            return 0.0, 0.0

        # surface gradient
        dz1 = 0.5 * float(self.dzgrid[0])
        G_surf = abs((float(self.Tgrid[0]) - self.Ts) / max(1e-6, dz1))

        # basal gradient over last 10 cm above soil
        zb = max(0.0, H - 0.10)  # depth from surface
        # interpolate T at depth zb
        Tz = float(np.interp(zb, self.zgrid, self.Tgrid, left=self.Tgrid[0], right=self.Tgrid[-1]))
        # soil is at depth H (from surface)
        # approximate gradient (Tsoil - T at 10cm above) over 0.10m
        # Note sign irrelevant for DH.
        # We'll store soil boundary temperature separately when updating; use last cell near basal as proxy:
        # Better: caller passes Tsoil. We'll instead estimate basal boundary as last cell + small.
        # Here: use last cell center as proxy for T just above soil; not perfect but OK.
        # We'll return a placeholder; actual DH uses caller's Tsoil for accuracy.
        G_basal = 0.0  # computed in update_regimes with Tsoil
        return G_surf, G_basal

    def update_regimes(self, forc: Forcing1H, dt: float):
        """
        Update x_dh and x_ws on layers using local gradients and wind threshold.
        """
        if not self.layers or self.Tgrid is None:
            return

        H = self.total_height()

        # basal gradient using soil boundary temperature
        zb = max(0.0, H - 0.10)
        Tz = float(np.interp(zb, self.zgrid, self.Tgrid, left=self.Tgrid[0], right=self.Tgrid[-1]))
        G_basal = abs((forc.Tsoil - Tz) / 0.10)

        # DH forcing
        dh_star = max(0.0, (G_basal - self.G0) / self.G0)

        # WS forcing (surface)
        Ut = self.Ut(forc.Ta)
        ws_star = max(0.0, (max(0.0, forc.U10 / max(1e-6, Ut)) ** self.p_ws) - 1.0)

        # Surface erodability proxy from top layer (very simple)
        top = self.layers[-1]
        # younger & lower density & not crusted => more erodible
        E_s = math.exp(-top.rho / 250.0) * (1.0 - top.x_ws)
        ws_star *= E_s

        # Apply DH to bottom layers (e.g., bottom 1-2 layers), WS to top 1-2 layers
        n = len(self.layers)
        bot_ids = list(range(0, min(2, n)))
        top_ids = list(range(max(0, n-2), n))

        for i in bot_ids:
            L = self.layers[i]
            # relaxation with memory
            L.x_dh += (dt / self.tau_dh) * (dh_star - L.x_dh)
            L.x_dh = float(np.clip(L.x_dh, 0.0, 1.0))

        for i in top_ids:
            L = self.layers[i]
            L.x_ws += (dt / self.tau_ws) * (ws_star - L.x_ws)
            L.x_ws = float(np.clip(L.x_ws, 0.0, 1.0))

        # Optionally: weak decay of regimes elsewhere
        for i in range(n):
            if i not in bot_ids:
                self.layers[i].x_dh *= math.exp(-dt / (10*self.tau_dh))
            if i not in top_ids:
                self.layers[i].x_ws *= math.exp(-dt / (10*self.tau_ws))

    # ---------
    # Albedo aging
    # ---------
    def update_albedo(self, dt: float, snowfall_swe: float):
        # simple relaxation from fresh to old; reset toward fresh with snowfall
        # snowfall_swe in kg/m2
        if snowfall_swe > 0.5:
            self.albedo = self.albedo_fresh
            return
        # exponential decay toward old
        self.albedo += (dt / self.albedo_tau) * (self.albedo_old - self.albedo)
        self.albedo = float(np.clip(self.albedo, 0.3, 0.95))

    # ---------
    # Create new layer on snowfall event
    # ---------
    def add_snowfall(self, swe_add: float):
        if swe_add <= 0:
            return
        if (not self.layers) or (swe_add >= self.min_new_layer_swe):
            # create a new top layer
            rho_new = self.rho_min
            self.layers.append(Layer(swe=swe_add, rho=rho_new))
        else:
            # merge into existing top layer
            top = self.layers[-1]
            # conserve thickness weighted by mass: update rho by mixing volumes
            # volume = mass/rho; new rho = (m1+m2)/(V1+V2)
            m1, m2 = top.swe, swe_add
            V1, V2 = m1/max(1e-6, top.rho), m2/max(1e-6, self.rho_min)
            top.swe = m1 + m2
            top.rho = (m1 + m2) / max(1e-12, (V1 + V2))
            top.age_s = 0.0

    # ---------
    # Compaction scheme: relax to target density
    # ---------
    def compaction_step(self, dt: float):
        if not self.layers:
            return

        # compute vertical stress sigma for each layer (Pa) from overburden
        # approximate sigma at layer center: weight of layers above
        # layers are bottom->top in list
        cum_mass_above = 0.0  # kg/m2
        sigmas = [0.0] * len(self.layers)
        for idx in range(len(self.layers)-1, -1, -1):  # top to bottom
            sigmas[idx] = G * cum_mass_above
            cum_mass_above += self.layers[idx].swe

        for i, L in enumerate(self.layers):
            sigma_i = sigmas[i]

            rho_grav = self.rho_min + (self.rho_ref - self.rho_min) * (1.0 - math.exp(-sigma_i / self.sigma0))
            rho_targ = rho_grav + self.dR_ws * L.x_ws - self.dR_dh * L.x_dh
            rho_targ = float(np.clip(rho_targ, self.rho_min, self.rho_max))

            # timescale
            tau = self.tau0 / (1.0 + self.a_sigma * (sigma_i / self.sigma0) + self.a_ws * L.x_ws)
            tau *= (1.0 + self.a_dh * L.x_dh)
            tau = max(1e3, tau)

            # implicit relaxation update
            L.rho = float(L.rho + (dt / (tau + dt)) * (rho_targ - L.rho))
            L.rho = float(np.clip(L.rho, self.rho_min, self.rho_max))

            L.age_s += dt

    # ---------
    # Layer fusion to keep Nmax and avoid tiny layers
    # ---------
    def fuse_layers(self):
        if len(self.layers) <= self.Nmax_layers:
            return
        # fuse the most similar adjacent pair (by rho and regime)
        def pair_cost(A: Layer, B: Layer) -> float:
            return (A.rho - B.rho)**2 + 5e4*(A.x_ws - B.x_ws)**2 + 5e4*(A.x_dh - B.x_dh)**2

        while len(self.layers) > self.Nmax_layers:
            costs = [pair_cost(self.layers[i], self.layers[i+1]) for i in range(len(self.layers)-1)]
            j = int(np.argmin(costs))
            A, B = self.layers[j], self.layers[j+1]
            # merge mass-conservatively
            mA, mB = A.swe, B.swe
            # volume mixing for density
            VA = mA / max(1e-6, A.rho)
            VB = mB / max(1e-6, B.rho)
            rho_new = (mA + mB) / max(1e-12, (VA + VB))
            swe_new = mA + mB
            # mix regimes by mass
            xws_new = (mA*A.x_ws + mB*B.x_ws) / max(1e-12, swe_new)
            xdh_new = (mA*A.x_dh + mB*B.x_dh) / max(1e-12, swe_new)
            age_new = min(A.age_s, B.age_s)  # toy choice
            self.layers[j] = Layer(swe=swe_new, rho=rho_new, x_ws=xws_new, x_dh=xdh_new, age_s=age_new)
            del self.layers[j+1]

    # ---------
    # Sublimation / deposition / erosion (toy)
    # Here we only remove mass by sublimation from SEB latent flux when LE<0 (upward, sublimation).
    # Deposition/erosion can be added later (using observed H change, or a transport proxy).
    # ---------
    def mass_update_from_latent(self, LE: float, dt: float):
        if not self.layers:
            return
        # Convention: In seb_residual we used LE = rho L C U (qa - qs).
        # If qa < qs -> LE negative -> cooling -> sublimation mass loss.
        if LE >= 0:
            return

        # mass flux (kg m-2 s-1) = -LE / L_sub
        mflux = (-LE) / L_SUB
        dm = mflux * dt
        dm = min(dm, self.total_swe())

        # remove from top layer first
        while dm > 0 and self.layers:
            top = self.layers[-1]
            take = min(dm, top.swe)
            top.swe -= take
            dm -= take
            if top.swe <= 1e-6:
                self.layers.pop()

    # ---------
    # One time step (dt seconds), with hourly forcing
    # ---------
    def step(self, forc: Forcing1H, dt: float = 3600.0, substeps: int = 1):
        """
        substeps: e.g., 1 (hourly) or 3-6 to stabilize near melt / strong wind.
        """
        dt_sub = dt / substeps
        for _ in range(substeps):
            # snowfall (SWE mass addition)
            swe_add = forc.Psnow * dt_sub
            if swe_add > 0:
                self.add_snowfall(swe_add)

            # update albedo
            self.update_albedo(dt_sub, swe_add)

            # rebuild thermal grid based on new height
            H = self.total_height()
            self.ensure_thermal_grid(H, forc.Tsoil)

            if self.Tgrid is None:
                self.time_s += dt_sub
                continue

            # solve SEB for Ts
            Ts = self.solve_Ts(forc)

            # melt constraint (toy): if Ts hits 0C, treat excess energy as melt later (not implemented)
            Ts = min(Ts, 273.15)
            self.Ts = Ts

            # conduction
            self.conduction_step(Ts=Ts, Tsoil=forc.Tsoil, dt=dt_sub)

            # compute fluxes at final Ts to get LE for sublimation
            T1 = float(self.Tgrid[0])
            rho1 = self.estimate_surface_density()
            k_half = k_snow(rho1)
            dz_half = 0.5 * float(self.dzgrid[0])
            F, Hflx, LEflx, Gflx, Rn = self.seb_residual(Ts, forc, k_half, T1, dz_half)

            # update DH/WS indices
            self.update_regimes(forc, dt_sub)

            # compaction toward target density
            self.compaction_step(dt_sub)

            # sublimation mass loss
            self.mass_update_from_latent(LEflx, dt_sub)

            # layer management
            self.fuse_layers()

            self.time_s += dt_sub

    # ---------
    # Diagnostics helpers
    # ---------
    def diagnostics(self) -> dict:
        return {
            "time_s": self.time_s,
            "H": self.total_height(),
            "SWE": self.total_swe(),
            "Ts": self.Ts,
            "albedo": self.albedo,
            "n_layers": len(self.layers),
            "layers": [(L.swe, L.thickness, L.rho, L.x_ws, L.x_dh) for L in self.layers]
        }

# -----------------------------
# Example usage (stub)
# -----------------------------
if __name__ == "__main__":
    dz_template = [0.01,0.01,0.02,0.02,0.03,0.03,0.04,0.04,0.05,0.05,0.06,0.06,0.08,0.08,0.10,0.10,0.12,0.12]

    model = ArcticSnowToy(dz_template=dz_template)

    # Make up 48 hours of forcing
    forcings = []
    for t in range(48):
        Ta = 258.15 + 2.0*math.sin(2*math.pi*(t/24.0))  # K
        RH = 0.7
        p  = 101325.0
        U10= 7.0 if t % 12 < 6 else 3.0
        SWd= 0.0
        LWd= 220.0
        Psnow = 0.0
        if t == 3:
            Psnow = 0.00002  # kg m-2 s-1 ~ 0.072 kg/m2/h = 0.072 mm SWE/h
        Tsoil = 268.15
        forcings.append(Forcing1H(Ta=Ta,RH=RH,p=p,U10=U10,SWd=SWd,LWd=LWd,Psnow=Psnow,Tsoil=Tsoil))

    for f in forcings:
        # subcycle a bit when windy
        sub = 3 if f.U10 > 8 else 1
        model.step(f, dt=3600.0, substeps=sub)
        d = model.diagnostics()
        print(d["time_s"]/3600, d["H"], d["SWE"], d["Ts"], d["n_layers"])
